In [1]:
# Install dependencies for the local VS Code kernel.
%pip install -q "transformers==4.57.1" accelerate bitsandbytes datasets huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import getpass
from huggingface_hub import HfApi, login, whoami

hf_token = getpass.getpass("Enter your Hugging Face read token: ")
login(token=hf_token, add_to_git_credential=False)
print("Logged in as:", whoami(token=hf_token)["name"])
HfApi().model_info(
    "meta-llama/Llama-3.1-8B-Instruct",
    token=hf_token,
)
print("Model access verified.")

In [ ]:
import gc
import os

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select the CUDA-enabled Python environment.")

dataset = load_dataset(
    "Saminx22/medical_data_for_slm",
    split="train[:100]",
)

teacher_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(teacher_id)
tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading Teacher Model with CPU offloading...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    quantization_config=quantization_config,
    device_map="auto",
    max_memory={0: "3500MiB", "cpu": "12GiB"},
    low_cpu_mem_usage=True,
)
teacher_model.eval()

saved_teacher_data = []
input_device = teacher_model.get_input_embeddings().weight.device
print(f"Processing {len(dataset)} examples on {input_device}...")

with torch.inference_mode():
    for index, example in enumerate(dataset):
        text = example["text"][:256]
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        )
        inputs = {name: value.to(input_device) for name, value in inputs.items()}

        outputs = teacher_model(**inputs, use_cache=False)
        logits = outputs.logits[0]
        topk_values, topk_indices = torch.topk(logits, k=100, dim=-1)

        saved_teacher_data.append({
            "input_ids": inputs["input_ids"][0].cpu(),
            "topk_logits": topk_values.cpu(),
            "topk_indices": topk_indices.cpu(),
        })

        del inputs, outputs, logits, topk_values, topk_indices
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if (index + 1) % 20 == 0:
            print(f"Processed {index + 1}/{len(dataset)} examples...")

file_name = "teacher_logits.pt"
torch.save(saved_teacher_data, file_name)
print(f"\nSaved {len(saved_teacher_data)} examples to {file_name}.")

del teacher_model
gc.collect()
torch.cuda.empty_cache()

Loading Teacher Model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing dataset and extracting logits...
Processed 20/100 examples...
Processed 40/100 examples...
Processed 60/100 examples...
Processed 80/100 examples...
Processed 100/100 examples...

Data saved successfully to teacher_logits.pt!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
import math
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

teacher_id = "meta-llama/Llama-3.1-8B-Instruct"

# 1. Load Tokenizer & 4-bit Quantized Teacher
tokenizer = AutoTokenizer.from_pretrained(teacher_id)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    quantization_config=quant_config,
    device_map="auto"
)
teacher_model.eval()

# 2. Load the identical 50 validation documents
val_data = load_dataset("Saminx22/medical_data_for_slm", split="validation[:50]")

total_loss = 0.0
total_batches = 0

with torch.no_grad():
    for item in val_data:
        text = item["text"][:512]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to("cuda")
        inputs["labels"] = inputs["input_ids"].clone()
        
        outputs = teacher_model(**inputs)
        total_loss += outputs.loss.item()
        total_batches += 1

teacher_val_loss = total_loss / total_batches
teacher_ppl = math.exp(teacher_val_loss)

print(f"Teacher Validation Loss: {teacher_val_loss:.4f}")
print(f"Teacher Validation Perplexity: {teacher_ppl:.2f}")